# Stage 12 submission: slide in, grade out

**The question:** what does the pipeline score on slides nobody has seen?

This notebook is the submission. For each test slide it runs the whole
pipeline live, exactly as in training: `get_tiles` (tissue mask, tiles; reads
only the image), `encode_tiles` (frozen UNI2-h, fp16, offline weights), then a
trained head's five models (`predict_ensemble`, their probabilities averaged).
Nothing is cached; the only file written is the prediction file.

- **The head** is set by `HEAD` below: `roformer_soft_labels` (RoPE model) or
  `abmil_soft_labels` (ABMIL), both trained on every slide in notebook 18. One
  submission per head.
- **No tiles** (an empty or failed slide) gives `FALLBACK_GRADE` 0, and is
  logged. So does an error on a slide, or running out of `TIME_BUDGET_H`.
- **No test folder** (locally, and in Kaggle's editor, where only the 3-row
  placeholder `test.csv` exists): the notebook writes the placeholder rows with
  grade 0, so the file is valid, and runs the pipeline on four training slides
  instead, to show it works and how long a slide takes. Locally it also checks
  that those live predictions equal the predictions from the stored features
  exactly.
- **Kaggle, internet off.** Attach, as datasets: the project's `src/` folder;
  the UNI2-h weights (`uni2h.pt`, private, CC-BY-NC-ND); the heads
  (`<head>.pt`); and the wheels (`openslide-python`, `openslide-bin`, `timm`
  1.0.29), installed below only if missing or a different version.

The hospital column of `test.csv` is never read: the pipeline sees the image
only.

## Setup

In [1]:
"""Install missing packages from attached wheels, then make the project's src/ folder importable.

1. On Kaggle (internet off), any wheel attached as a dataset is installed if its
   package is missing or at another version: openslide-python and openslide-bin
   (the tiling reads slides with them) and timm (UNI2-h must be built by the
   same version as in training). Locally nothing is installed.
2. src/ is one folder up locally; on Kaggle it is searched for at any depth
   under /kaggle/input, since Kaggle nests attached datasets. The notebook stops
   with a clear message if it is missing.
"""

import glob
import importlib.metadata
import os
import subprocess
import sys

WANTED = {"openslide-python": "1.4.6", "openslide-bin": "4.0.1.2", "timm": "1.0.29"}


def installed(package):
    try:
        return importlib.metadata.version(package)
    except importlib.metadata.PackageNotFoundError:
        return None


if os.path.exists("/kaggle/input"):
    wheels = sorted(glob.glob("/kaggle/input/**/*.whl", recursive=True))
    to_install = []
    for package, version in WANTED.items():
        if installed(package) != version:
            prefix = package.replace("-", "_") + "-" + version + "-"
            to_install += [w for w in wheels if os.path.basename(w).startswith(prefix)]
    if to_install:
        subprocess.run([sys.executable, "-m", "pip", "install", "--no-index", "--no-deps", "--quiet", *to_install],
                       check=False)
print({package: installed(package) for package in WANTED})

candidates = [os.path.abspath("..")] + sorted(
    os.path.dirname(os.path.dirname(p)) for p in glob.glob("/kaggle/input/**/src/tiling.py", recursive=True))
SRC_ROOT = next((root for root in candidates if os.path.isfile(os.path.join(root, "src", "tiling.py"))), None)
if SRC_ROOT is None:
    raise FileNotFoundError("src/ not found; on Kaggle, attach the project's src folder as a dataset")
if SRC_ROOT not in sys.path:
    sys.path.insert(0, SRC_ROOT)
print(f"src from {SRC_ROOT}")

{'openslide-python': '1.4.6', 'openslide-bin': '4.0.1.2', 'timm': '1.0.29'}
src from /home/aztrek/projects/Prostate-Cancer-Grading


In [2]:
"""Imports.

os, tempfile, time    paths, where outputs go locally, the time budget.
numpy, pandas         probabilities, test.csv and the prediction file.
torch                 the GPU name.
src.encode            the offline encoder and the one pixel transform.
src.features          the packed training features, for the local check only.
src.mil               the trained heads and the tile-position rule.
src.paths             the competition data, the weights and the heads, locally and on Kaggle.
src.tiling            the tiling: slide file in, tiles and level-0 coordinates out.
"""

import os
import tempfile
import time

import numpy as np
import pandas as pd
import torch

from src.encode import (
    encode_tiles,  # tiles in, fp16 features [n, 1536] out; the only place the pixel transform lives
    load_encoder_offline,  # UNI2-h from the weights file, no network
    prepare,  # eval mode, fp16 on the GPU
)
from src.features import (
    open_packed,  # the packed training features, for the local check only
)
from src.mil import (
    HEAD_NEEDS_POSITIONS,  # whether a head needs tile positions (the RoPE head does)
    grid_positions,  # level-0 coordinates to grid positions: the same rule as in training
    load_ensemble,  # a head's five models and their record
    predict_ensemble,  # one slide's grade from the five models' averaged probabilities
)
from src.paths import (
    DATA_DIR,  # the competition data, locally or on Kaggle
    ON_KAGGLE,  # True on Kaggle
    head_path,  # a head's file: data/models/ locally, an attached dataset on Kaggle
    slide_path,  # a training slide's .tiff, for the check slides
    weights_path,  # the UNI2-h weights: local file, or the private dataset on Kaggle
)
from src.tiling import (
    get_tiles,  # the tiling, exactly as in training
)

In [3]:
"""Settings.

1. HEAD: which trained head this submission uses.
2. FALLBACK_GRADE: the grade for a slide with no tiles, an error, or no time left.
3. TIME_BUDGET_H: stop predicting after this and give the rest the fallback, so
   the file is always complete. Kaggle's limit for this competition has not
   been confirmed; 8 h leaves margin under the 9 h assumed in notebook 7.
4. CHECK_SLIDES: four training slides (the reproduction test's) predicted when
   there is no test folder.
5. OUT_DIR: /kaggle/working on Kaggle (where submission.csv must be); the system
   temporary folder locally, so nothing lands in the repository.
"""

HEAD = "roformer_soft_labels"
FALLBACK_GRADE = 0
TIME_BUDGET_H = 8.0
CHECK_SLIDES = ["1836f6539ccc9e37d426603cc4526f8b", "4726ffb66a6516baff6de98df2b5a26b",
                "0240270d611683829ea5bee2a8c52eb0", "03b113d63b06f6740d82b4cfa1ee4a75"]
OUT_DIR = "/kaggle/working" if ON_KAGGLE else tempfile.gettempdir()

test = pd.read_csv(os.path.join(DATA_DIR, "test.csv"))
test_dir = os.path.join(DATA_DIR, "test_images")
has_test = os.path.isdir(test_dir) and len(os.listdir(test_dir)) > 0
print(f"test.csv: {len(test)} rows; test images: {'present' if has_test else 'absent (check mode)'}; "
      f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none'}")

test.csv: 3 rows; test images: absent (check mode); GPU: NVIDIA GeForce RTX 4060


In [4]:
"""Load the encoder and the head's models."""

started = time.perf_counter()
encoder = prepare(load_encoder_offline(weights_path()))
models, record = load_ensemble(head_path(HEAD))
head, loss = record["head"], record["loss"]
needs_positions = HEAD_NEEDS_POSITIONS[head]
print(f"{HEAD}: {len(models)} models ({head}, {loss}, seeds {record['seeds']}), trained on "
      f"{record['info']['training_slides']:,} slides of {record['info']['feature_set']}; "
      f"loaded in {time.perf_counter() - started:.0f} s")

roformer_soft_labels: 5 models (roformer, soft_labels, seeds [0, 1, 2, 3, 4]), trained on 10,615 slides of uni2h-L1-224-t144-hsv-v1; loaded in 5 s


## Predict

One slide at a time: tiles, features, the five models' averaged
probabilities, the grade. Every slide gets a row; anything that stops a slide
from being predicted is recorded in the `status` column.

In [5]:
"""Predict every slide (test slides, or the check slides), then write the prediction file.

predict_slide(path):
1. get_tiles on the slide file; no tiles gives FALLBACK_GRADE with status "no tiles".
2. encode_tiles; grid positions from the tiles' level-0 coordinates (RoPE head).
3. predict_ensemble: the averaged probabilities and the grade.
Errors give FALLBACK_GRADE with the error as status; once TIME_BUDGET_H has
passed, the remaining slides get FALLBACK_GRADE with status "out of time".
"""


def predict_slide(path):
    tiles, coords, _ = get_tiles(path)
    if not tiles:
        return FALLBACK_GRADE, None, 0, "no tiles"
    features = encode_tiles(encoder, tiles)
    positions = grid_positions(coords) if needs_positions else None
    grade, probabilities = predict_ensemble(models, loss, features, positions)
    return grade, probabilities, len(tiles), "ok"


if has_test:
    jobs = [(i, os.path.join(test_dir, f"{i}.tiff")) for i in test["image_id"]]
else:
    jobs = [(i, slide_path(i)) for i in CHECK_SLIDES]

rows = []
started = time.perf_counter()
for image_id, path in jobs:
    t0 = time.perf_counter()
    if time.perf_counter() - started > TIME_BUDGET_H * 3600:
        grade, probabilities, n, status = FALLBACK_GRADE, None, 0, "out of time"
    else:
        try:
            grade, probabilities, n, status = predict_slide(path)
        except Exception as error:  # noqa: BLE001 -- recorded in status; every slide must get a row
            grade, probabilities, n, status = FALLBACK_GRADE, None, 0, f"error: {error!r}"
    rows.append({"image_id": image_id, "isup_grade": grade, "tiles": n, "status": status,
                 "seconds": round(time.perf_counter() - t0, 2),
                 "probabilities": probabilities})
details = pd.DataFrame(rows)

if has_test:
    submission = details[["image_id", "isup_grade"]]
else:
    submission = test[["image_id"]].assign(isup_grade=FALLBACK_GRADE)
submission.to_csv(os.path.join(OUT_DIR, "submission.csv"), index=False)
details.to_csv(os.path.join(OUT_DIR, "predictions_detailed.csv"), index=False)

print(details.drop(columns="probabilities").to_string(index=False))
print(f"\n{len(details)} slides in {(time.perf_counter() - started) / 60:.1f} min; "
      f"status counts {details['status'].str.split(':').str[0].value_counts().to_dict()}; "
      f"median {details['seconds'].median():.2f} s per slide")
print(f"wrote {os.path.join(OUT_DIR, 'submission.csv')} ({len(submission)} rows)")

                        image_id  isup_grade  tiles status  seconds
1836f6539ccc9e37d426603cc4526f8b           1    144     ok     2.58
4726ffb66a6516baff6de98df2b5a26b           2    144     ok     2.40
0240270d611683829ea5bee2a8c52eb0           3     59     ok     0.96
03b113d63b06f6740d82b4cfa1ee4a75           4     70     ok     1.31

4 slides in 0.1 min; status counts {'ok': 4}; median 1.85 s per slide
wrote /tmp/submission.csv (3 rows)


## Check mode: do live predictions equal the stored ones?

Only without test images, and only where the packed training features exist
(locally). The check slides' tiles and features were stored by the extraction;
the reproduction test showed live tiling and encoding reproduce them bit for
bit on the same GPU, so the live grades and probabilities must equal those
computed from the stored features exactly.

In [6]:
"""Compare the live check-slide predictions with predictions from the stored features (local only)."""

feature_set = record["info"]["feature_set"]
if has_test:
    print("test images present: no check")
else:
    try:
        packed = open_packed(feature_set)
    except FileNotFoundError:
        packed = None
        print(f"{feature_set} is not available here (as on Kaggle): no comparison, the live run above is the check")
    if packed is not None:
        where = dict(zip(packed.index["slide_id"], zip(packed.starts, packed.ends)))
        xy = packed.tiles[["x", "y"]].to_numpy()
        for row in rows:
            start, end = where[row["image_id"]]
            stored = torch.from_numpy(np.array(packed.features[start:end]))
            grade, probabilities = predict_ensemble(models, loss, stored, grid_positions(xy[start:end]) if needs_positions else None)
            same = grade == row["isup_grade"] and np.array_equal(probabilities, row["probabilities"])
            print(f"{row['image_id']}: live {row['isup_grade']}, stored {grade} -> {'identical' if same else 'DIFFERENT'}")

1836f6539ccc9e37d426603cc4526f8b: live 1, stored 1 -> identical
4726ffb66a6516baff6de98df2b5a26b: live 2, stored 2 -> identical
0240270d611683829ea5bee2a8c52eb0: live 3, stored 3 -> identical
03b113d63b06f6740d82b4cfa1ee4a75: live 4, stored 4 -> identical


## Result

Run locally 2026-11-19 on the RTX 4060 PC, in check mode (no test images here),
with `HEAD = "roformer_soft_labels"`.

- Packages as in training (openslide-python 1.4.6, openslide-bin 4.0.1.2, timm
  1.0.29); encoder and the five models loaded in 5 s.
- The four check slides went through the whole live pipeline, `.tiff` to grade:
  all predicted, median 1.85 s per slide (0.96 to 2.58; 59 to 144 tiles).
- **Live equals stored:** each slide's grade and probabilities are identical to
  those from its stored features, for the RoPE model here and for ABMIL (same
  check run separately). So the submission computes exactly what the models
  were trained on.
- `submission.csv` held the three placeholder rows of `test.csv`, grade 0.

Not yet run on Kaggle. There it will also show Kaggle's GPU time per slide and
whether the wheels install offline.